# Post-COVID Video Games Worldwide
- 캐글 뒤져보다가 또 발견했습니다... 예.
- 팀플 데이터도 찾아야되는데 하... 엘든링 말레니아 깨는게 이거보다 더 쉬울듯합니다...
- 데이터셋은 [여기](https://www.kaggle.com/datasets/otegbolamarvellous/post-covid-video-games-worldwide-2021-2025)서 받았습니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="icefire", style="darkgrid", font_scale=1)
sns.color_palette("tab10", as_cmap=True)

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Umdot 12'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 9
plt.rcParams['font.size'] = 16
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
meta_df = pd.read_csv('data/metacritic_games_data.csv')

# 데이터프레임 정보 확인

## .info()

In [ ]:
meta_df.info()

## .describe()

In [ ]:
meta_df.describe()

In [ ]:
meta_df.describe(include='O')

## .isna().sum()

In [ ]:
meta_df.isna().sum()

## .columns

In [ ]:
meta_df.columns

## .head()

In [ ]:
meta_df.head()

# 전처리이이아

## 결측값 우짜꼬...

In [ ]:
meta_df_na = meta_df.query('User_Score.isna()').index

meta_df_na

In [ ]:
# 결측값이 어디 들어있나 확인
meta_df.loc[meta_df_na]

In [ ]:
meta_df['User_Score'] = pd.to_numeric(meta_df['User_Score'], errors='coerce')

## 플랫폼 범주화

In [ ]:
# 1. 콤마(,)로 나누고(split), 양쪽 공백을 제거(strip)한 리스트의 길이를 측정
meta_df['Platform_classification'] = meta_df['Platform'].apply(
    lambda x: 'Singleplatform' if len(str(x).split(',')) == 1 else 'Multiplatform'
)

# 결과 확인
print(meta_df['Platform_classification'].value_counts())

## 년도만 따로 추출

In [ ]:
# 일단 datetime으로 변환부터 합시다
meta_df['Release_Date'] = pd.to_datetime(meta_df['Release_Date'], dayfirst=True, errors='coerce')
meta_df['Year'] = meta_df['Release_Date'].dt.year # 응 년도 만들거야

In [ ]:
meta_df['Year'].value_counts()

## PC released?

In [ ]:
# 'PC'가 문자열 안에 들어있으면 True, 없으면 False를 반환합니다.
meta_df['PC_included'] = meta_df['Platform'].apply(
    lambda x: 'PC_included' if 'PC' in str(x) else 'PC_unincluded'
)

# 결과 확인
print(meta_df['PC_included'].value_counts())

## 평점에 따른 분류
### 전문가 평점
- 90점 이상: 대부분의 극찬
- 75~89점: 전반적인 호평
- 50~74점: 호불호 갈리거나 보통
- 20~49점: 전반적인 혹평
- 0~19점: 똥

~~이거 람다 못쓰겠는데~~

In [ ]:
def metascore_classificaion(score):
    if score >= 90:
        return 'Universal acclaim'
    elif score >= 75:
        return 'Generally favorable reviews'
    elif score >= 50:
        return 'Mixed or average reviews'
    elif score >= 20:
        return 'Generally unfavorable reviews'
    else:
        return 'Overwhelming dislike'

def userscore_classificaion(score):
    if score >= 9:
        return 'Universal acclaim'
    elif score >= 7.5:
        return 'Generally favorable reviews'
    elif score >= 5:
        return 'Mixed or average reviews'
    elif score >= 2:
        return 'Generally unfavorable reviews'
    else:
        return 'Overwhelming dislike'

In [ ]:
meta_df['Metascore_classification'] = meta_df['Metascore'].apply(metascore_classificaion)

In [ ]:
meta_df['Metascore_classification']

### 유저 스코어 범주화
- 9점 이상: 대부분의 극찬
- 7.5~8.9점: 전반적인 호평
- 5.0~7.4점: 호불호 갈리거나 보통
- 2.0~4.9점: 전반적인 혹평
- 0~1.9점: 똥

In [ ]:
meta_df['Userscore_classification'] = meta_df['User_Score'].apply(userscore_classificaion)

In [ ]:
meta_df['Userscore_classification']

# 좋아 드가자

## 갓겜과 똥겜

### 메타크리틱 스코어 90점 이상

In [ ]:
# 일단 메타크리틱 평점이 90점 이상인 것만 따로 뻅시다.
meta_god_df = meta_df.query('Metascore >= 90')

meta_god_df

In [ ]:
# 평론가와 유저가 다 만족한 게임이 있을까?
meta_god_df.query('User_Score >= 9')

In [ ]:
# 평론가와 유저 반응이 다른 게임
meta_god_df.query('User_Score < 9')

## 메타크리틱 스코어 20점 미만

In [ ]:
# DDONG게임
meta_ddong_df = meta_df.query('Metascore < 20')

meta_ddong_df

- 없는데요?

In [ ]:
meta_ddong_df = meta_df.query('User_Score < 2')

meta_ddong_df

In [ ]:
# 50점 미만은 꽤 있다.
meta_ddong_df = meta_df.query('Metascore < 50')

meta_ddong_df

In [ ]:
meta_ddong_df.query('User_Score < 2')

- 옵치 2의 악명은 나도 익히 들어서 알지...

## 플랫폼이 하나인가? 아니면 여러개인가?

In [ ]:
meta_df.groupby(['Platform_classification']).size()

- 그치... 요즘 어지간하면 다 멀티로 나오지...

In [ ]:
meta_df.groupby(['Platform_classification','PC_included']).size()

### 시각화

In [ ]:
sns.countplot(data=meta_df, x='Platform_classification', hue='PC_included', palette="icefire")
plt.legend(['PC판 미발매', 'PC판 발매'])
plt.xlabel('단일 플랫폼 여부 및 PC판 발매 여부')
plt.ylabel('발매 게임 수')
plt.title('단일 플랫폼 여부 및 PC판 발매여부에 따른 게임 수', fontsize=20)
plt.show()

## 발매년도 추가

In [ ]:
meta_df.groupby(['Year','Platform_classification','PC_included']).size()

In [ ]:
meta_year = meta_df.groupby(['Year','Platform_classification','PC_included']).size().unstack()
meta_year['Sum'] = meta_year['PC_included'] + meta_year['PC_unincluded']

### 시각화

In [ ]:
sns.barplot(data=meta_year, x='Platform_classification', y = 'Sum', hue='Year', palette="icefire")
plt.xlabel('단일 플랫폼 여부 및 PC판 발매 여부')
plt.ylabel('발매 게임 수')
plt.title('단일 플랫폼 여부 및 PC판 발매여부에 따른 게임 수', fontsize=20)
plt.show()

## 닌텐도 게임도 과연 멀티플랫폼이 있을까?

In [ ]:
nintendo_df = meta_df.query("Publisher.str.contains('Nintendo')", engine='python')
nintendo_df

In [ ]:
sns.countplot(nintendo_df, x = 'Platform_classification', hue='Platform_classification', palette="icefire", legend=True)
plt.title('닌텐도도 멀티플랫폼이 있나요?', fontsize=20)
plt.xlabel('멀티플랫폼 여부')
plt.ylabel('발매 게임 수')
plt.legend()
plt.show()

In [ ]:
nintendo_df.query('Platform_classification == "Multiplatform"')

## 년도별 스코어 최댓값
### 메타스코어

In [ ]:
meta_god_loc = meta_df.groupby('Year')['Metascore'].idxmax()
meta_df.loc[meta_god_loc][['Title', 'Metascore', 'User_Score']].reset_index(drop=True)

- 와 엘든링...

### 유저 스코어

In [ ]:
meta_god_loc = meta_df.groupby('Year')['User_Score'].idxmax()
meta_df.loc[meta_god_loc][['Title', 'Metascore', 'User_Score']].reset_index(drop=True)

- 아 발더스 게이트는 킹정이지.

## 년도별 스코어 최솟값
### 메타스코어

In [ ]:
meta_ddong_loc = meta_df.groupby('Year')['Metascore'].idxmin()
meta_df.loc[meta_ddong_loc][['Title', 'Metascore', 'User_Score']].reset_index(drop=True)

### 유저 스코어

In [ ]:
meta_ddong_loc = meta_df.groupby('Year')['User_Score'].idxmin()
meta_df.loc[meta_ddong_loc][['Title', 'Metascore', 'User_Score']].reset_index(drop=True)

## 장르별 스코어
### 메타스코어 (스코어 오름차순)

In [ ]:
meta_genre_loc = meta_df.groupby('Genre')['Metascore'].idxmax()
meta_df.loc[meta_genre_loc][['Genre', 'Title', 'Metascore', 'User_Score']].reset_index(drop=True).sort_values('Metascore', ascending=False)

### 유저 스코어

In [ ]:
best_idx = meta_df.dropna(subset=['User_Score']).groupby('Genre')['User_Score'].idxmax()
best_genre_df = meta_df.loc[best_idx]

best_genre_df[['Genre', 'Title', 'Metascore', 'User_Score']].reset_index(drop=True).sort_values('User_Score', ascending=False)

### 메타스코어 (DDONG게임순)

In [ ]:
meta_genre_loc = meta_df.groupby('Genre')['Metascore'].idxmin()
meta_df.loc[meta_genre_loc][['Genre', 'Title', 'Metascore', 'User_Score']].reset_index(drop=True).sort_values('Metascore')

### 유저스코어 (DDONG게임순)

In [ ]:
best_idx = meta_df.dropna(subset=['User_Score']).groupby('Genre')['User_Score'].idxmin()
best_genre_df = meta_df.loc[best_idx]

best_genre_df[['Genre', 'Title', 'Metascore', 'User_Score']].reset_index(drop=True).sort_values('User_Score')

## 리뷰가 많은 게임
### 비평가 리뷰가 많은 순

In [ ]:
most_review = meta_df.sort_values('Critic_Review_Count', ascending=False)
most_review[['Title', 'Critic_Review_Count', 'User_Review_Count', 'Metascore', 'User_Score']]

In [ ]:
# 유저 리뷰가 5000개 이상인 게임들만 골라봤음
most_review = meta_df.sort_values('Critic_Review_Count', ascending=False)
most_review.query('User_Review_Count >= 5000')[['Title', 'Critic_Review_Count', 'User_Review_Count', 'Metascore', 'User_Score']]

### 유저 리뷰가 많은 순

In [ ]:
most_review = meta_df.sort_values('User_Review_Count', ascending=False)
most_review[['Title', 'Critic_Review_Count', 'User_Review_Count', 'Metascore', 'User_Score']]

In [ ]:
# 비평가 리뷰가 100개 이상인 게임
most_review = meta_df.sort_values('User_Review_Count', ascending=False)
most_review.query('Critic_Review_Count >= 100')[['Title', 'Critic_Review_Count', 'User_Review_Count', 'Metascore', 'User_Score']]